In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4.1-nano")
output_parser = StrOutputParser()


In [8]:
from dotenv import load_dotenv
import os
load_dotenv()
# os.getenv('OPENAI_API_KEY')

True

In [9]:
llm.invoke("Where is the capital of Korea? Answer me in Korean")

AIMessage(content='한국의 수도는 서울입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 18, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_38343a2f8f', 'id': 'chatcmpl-BmX1gJNJebSSpf98Pa0gYOoG5KF52', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--db1d7eba-7591-403a-9522-70d5b15a1894-0', usage_metadata={'input_tokens': 18, 'output_tokens': 7, 'total_tokens': 25, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# 1. 나라 -> 음식 추천

In [34]:
food_prompt = PromptTemplate(
    template="""
    Please recommend the most famous food from {country}.    
    When answering, write full name of country and,
    please use the national flag and food of the answer as an emoji.
    And give a short explanation of the name of the food and one line.

    If the user wants a specific format, follow this instruction: {format}
    """,
    input_variables=["country", "format"]
)


food_chain = food_prompt | llm | output_parser

output_parser.invoke(llm.invoke(food_prompt.invoke({"country":"Japan", "format": "Just the food name only"})))

'🇯🇵 Japan - 🍣 Sushi  \nSushi\'s name means "sour rice" in Japanese, referring to the vinegared rice combined with raw fish or other ingredients, and it is a beloved traditional dish worldwide.'

# 2. 음식 -> 레시피

In [38]:
recipe_prompt = PromptTemplate(
    template="""
    Write a detailed recipe for the food: {food}.
    Include ingredients and cooking instructions.
    When answering, please use the national flag and food of the answer as an emoji.

    If the user wants a specific output style, follow this instruction: {style}
    """,
    input_variables=["food", "style"]
)


recipe_chain = recipe_prompt | llm | output_parser
output_parser.invoke(llm.invoke(recipe_prompt.invoke({"food":"kimchi", "style": "use emoji of food"})))

'🇰🇷🥢 Kimchi Recipe\n\n**Ingredients:**\n- 1 medium Napa cabbage (about 2-3 pounds)\n- 1/4 cup sea salt\n- 4 cups water\n- 1 tablespoon grated ginger\n- 4 garlic cloves, minced\n- 3 tablespoons fish sauce or soy sauce (for vegetarian)\n- 1 tablespoon sugar\n- 3-4 tablespoons Korean red pepper flakes (gochugaru)\n- 4 scallions, chopped\n- 1 small carrot, julienned (optional)\n- 1/4 medium daikon radish, julienned (optional)\n\n**Instructions:**\n1. **Prepare the cabbage:** Cut the Napa cabbage lengthwise into halves or quarters, then into bite-sized pieces. Place the cabbage in a large mixing bowl and sprinkle evenly with sea salt. Mix thoroughly to coat all pieces.\n2. **Salting:** Pour water over the salted cabbage, ensuring all pieces are submerged. Place a weight on top to keep submerged. Leave it to sit at room temperature for 1.5 to 2 hours, tossing occasionally.\n3. **Rinse and drain:** After salting, rinse the cabbage thoroughly under cold water multiple times to remove excess sa

# 3. 체인 연결

In [43]:
from langchain_core.runnables import RunnablePassthrough
final_chain = {"country": RunnablePassthrough(), "format": RunnablePassthrough()} | {"food": food_chain
} | {"food": RunnablePassthrough(), "style": lambda x: x["format"] } | recipe_chain


# 4. 출력

In [44]:
result = final_chain.invoke({
    "country": "Japan",
    "format": "Summarize the recipe in 3 steps with emojis"
})
print(result)


🇯🇵🍣

**Ingredients:**
- Sushi rice (2 cups)
- Rice vinegar (1/4 cup)
- Sugar (2 tbsp)
- Salt (1 tsp)
- Nori sheets
- Fresh fish (salmon, tuna) or cooked shrimp
- Vegetables like cucumber, avocado
- Soy sauce, wasabi, and pickled ginger for serving

**Steps:**
1. 🍚⭐ Cook the sushi rice, then gently mix in rice vinegar, sugar, and salt. Let it cool.
2. 🥢🌀 Spread rice on nori, add fish and vegetables, then roll tightly using a bamboo mat.
3. 🍣✂️ Slice the sushi roll, serve with soy sauce, wasabi, and pickled ginger.

Enjoy your homemade sushi!
